# Result Error Analysis (DL19)

This notebook starts with **DL19 (TREC run)** error analysis over data in `all_results`, following the requested workflow:

1. Load official qrels (ground-truth relevance labels)
2. Compare top-k ranking differences across methods from TREC run files
3. Identify failure queries where HyDE underperforms the baseline and print interpretable diagnostics


In [1]:
from __future__ import annotations

import json
import math
from collections import defaultdict
from pathlib import Path
from statistics import mean
from urllib.request import urlretrieve

# Optional: if pandas is available, use DataFrame display; otherwise, the analysis still runs.
try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path('.').resolve()
DL19_DIR = ROOT / 'all_results' / 'dl19'

RUN_PATHS = {
    'bm25': DL19_DIR / 'bm25' / 'run.trec',
    'contriever': DL19_DIR / 'contriever' / 'run.trec',
    'hyde': DL19_DIR / 'hyde' / 'run.trec',
}

BASELINE_NAME = 'bm25'
HYDE_NAME = 'hyde'
TOP_K = 10


## Step 1: Load Relevance Judgments (qrels)

Use local qrels first; if missing, automatically download official DL19 files from the Anserini tools repository.


In [2]:
QRELS_URL = 'https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/qrels.dl19-passage.txt'
TOPICS_URL = 'https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.dl19-passage.txt'

QRELS_PATH = DL19_DIR / 'qrels.dl19-passage.txt'
TOPICS_PATH = DL19_DIR / 'topics.dl19-passage.txt'


def ensure_file(path: Path, download_url: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f'[download] {download_url} -> {path}')
        urlretrieve(download_url, path)
    else:
        print(f'[cached] {path}')
    return path


def parse_qrels(qrels_path: Path):
    qrels = defaultdict(dict)
    with qrels_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            qid, _, docid, rel = line.split()
            qrels[qid][docid] = int(rel)
    return dict(qrels)


def parse_topics(topics_path: Path):
    topics = {}
    with topics_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            # Format: qid<TAB>query
            qid, query = line.split('\t', 1)
            topics[qid] = query
    return topics


qrels_path = ensure_file(QRELS_PATH, QRELS_URL)
topics_path = ensure_file(TOPICS_PATH, TOPICS_URL)

qrels = parse_qrels(qrels_path)
topics = parse_topics(topics_path)

print(f'qrels queries: {len(qrels)}')
print(f'topics queries: {len(topics)}')


[cached] /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3/baseline_reproduction/ANLP-HW34/all_results/dl19/qrels.dl19-passage.txt
[download] https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.dl19-passage.txt -> /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3/baseline_reproduction/ANLP-HW34/all_results/dl19/topics.dl19-passage.txt
qrels queries: 43
topics queries: 43


## Step 2: Compare Ranked Retrieval Results

Read TREC run files for each method, build per-query ranked lists, then compute per-query metrics for baseline vs. HyDE.


In [4]:
def parse_trec_run(run_path: Path):
    # Returns: {qid: [docid1, docid2, ...]} sorted by ascending rank
    grouped = defaultdict(list)
    with run_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            qid, _q0, docid, rank, score, tag = line.split()
            grouped[qid].append((int(rank), docid, float(score), tag))

    runs = {}
    for qid, rows in grouped.items():
        rows.sort(key=lambda x: x[0])
        runs[qid] = [docid for _, docid, _, _ in rows]
    return runs


runs = {}
for name, path in RUN_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing run file: {path}')
    runs[name] = parse_trec_run(path)
    print(f'{name:10s} | queries: {len(runs[name])} | file: {path}')

shared_qids = sorted(set(qrels) & set(runs[BASELINE_NAME]) & set(runs[HYDE_NAME]))
print(f'\nShared qids (qrels & {BASELINE_NAME} & {HYDE_NAME}): {len(shared_qids)}')
print('Sample qids:', shared_qids[:5])


bm25       | queries: 43 | file: /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3/baseline_reproduction/ANLP-HW34/all_results/dl19/bm25/run.trec
contriever | queries: 43 | file: /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3/baseline_reproduction/ANLP-HW34/all_results/dl19/contriever/run.trec
hyde       | queries: 43 | file: /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3/baseline_reproduction/ANLP-HW34/all_results/dl19/hyde/run.trec

Shared qids (qrels & bm25 & hyde): 43
Sample qids: ['1037798', '104861', '1063750', '1103812', '1106007']


In [5]:
def ndcg_at_k(ranked_docids, qrels_for_qid, k=10):
    gains = [qrels_for_qid.get(docid, 0) for docid in ranked_docids[:k]]
    dcg = sum(((2 ** rel) - 1) / math.log2(i + 2) for i, rel in enumerate(gains))

    ideal_rels = sorted(qrels_for_qid.values(), reverse=True)[:k]
    idcg = sum(((2 ** rel) - 1) / math.log2(i + 2) for i, rel in enumerate(ideal_rels))
    return (dcg / idcg) if idcg > 0 else 0.0


def recall_at_k(ranked_docids, qrels_for_qid, k=10):
    relevant = {docid for docid, rel in qrels_for_qid.items() if rel > 0}
    if not relevant:
        return 0.0
    retrieved = set(ranked_docids[:k])
    return len(retrieved & relevant) / len(relevant)


comparison_rows = []
for qid in shared_qids:
    qrels_for_qid = qrels[qid]
    baseline_ranked = runs[BASELINE_NAME][qid]
    hyde_ranked = runs[HYDE_NAME][qid]

    baseline_ndcg10 = ndcg_at_k(baseline_ranked, qrels_for_qid, k=TOP_K)
    hyde_ndcg10 = ndcg_at_k(hyde_ranked, qrels_for_qid, k=TOP_K)

    baseline_recall10 = recall_at_k(baseline_ranked, qrels_for_qid, k=TOP_K)
    hyde_recall10 = recall_at_k(hyde_ranked, qrels_for_qid, k=TOP_K)

    comparison_rows.append({
        'qid': qid,
        'query': topics.get(qid, ''),
        'baseline_ndcg10': baseline_ndcg10,
        'hyde_ndcg10': hyde_ndcg10,
        'delta_ndcg10': hyde_ndcg10 - baseline_ndcg10,
        'baseline_recall10': baseline_recall10,
        'hyde_recall10': hyde_recall10,
        'delta_recall10': hyde_recall10 - baseline_recall10,
    })

comparison_by_qid = {row['qid']: row for row in comparison_rows}
print(f'Per-query comparison rows: {len(comparison_rows)}')

if pd is not None:
    display(
        pd.DataFrame(comparison_rows)
        .sort_values('delta_ndcg10')
        .head(10)
    )
else:
    worst_10 = sorted(comparison_rows, key=lambda x: x['delta_ndcg10'])[:10]
    for row in worst_10:
        print(row['qid'], f"delta_ndcg10={row['delta_ndcg10']:.4f}", row['query'])


Per-query comparison rows: 43


,qid,query,baseline_ndcg10,hyde_ndcg10,delta_ndcg10,baseline_recall10,hyde_recall10,delta_recall10
0,1037798,who is robert gray,0.381611,0.000000,-0.381611,0.076923,0.000000,-0.076923
20,148538,difference between rn and bsn,0.368889,0.030589,-0.338300,0.049505,0.019802,-0.029703
25,19335,anthropological definition of environment,0.605109,0.272580,-0.332529,0.200000,0.100000,-0.100000
33,489204,right pelvic pain causes,0.314115,0.021942,-0.292173,0.041667,0.010417,-0.031250
38,855410,what is theraderm used for,0.960413,0.692615,-0.267798,1.000000,1.000000,0.000000
32,47923,axon terminals or synaptic knob definition,0.356604,0.098995,-0.257609,0.089286,0.026786,-0.062500
40,87452,causes of military suicide,0.320127,0.078418,-0.241710,0.098765,0.024691,-0.074074
39,87181,causes of left ventricular hypertrophy,0.589665,0.349915,-0.239750,0.096386,0.060241,-0.036145
1,104861,cost of interior concrete flooring,0.810750,0.593060,-0.217690,0.056738,0.042553,-0.014184
21,156493,do goldfish grow,0.848748,0.651123,-0.197625,0.075188,0.052632,-0.022556


In [6]:
def summarize_metric(rows, metric_name):
    return mean(row[metric_name] for row in rows) if rows else float('nan')


summary = {
    f'{BASELINE_NAME}_mean_ndcg10': summarize_metric(comparison_rows, 'baseline_ndcg10'),
    f'{HYDE_NAME}_mean_ndcg10': summarize_metric(comparison_rows, 'hyde_ndcg10'),
    f'{BASELINE_NAME}_mean_recall10': summarize_metric(comparison_rows, 'baseline_recall10'),
    f'{HYDE_NAME}_mean_recall10': summarize_metric(comparison_rows, 'hyde_recall10'),
}

wins_ndcg = sum(1 for r in comparison_rows if r['delta_ndcg10'] > 0)
losses_ndcg = sum(1 for r in comparison_rows if r['delta_ndcg10'] < 0)
ties_ndcg = len(comparison_rows) - wins_ndcg - losses_ndcg

wins_recall = sum(1 for r in comparison_rows if r['delta_recall10'] > 0)
losses_recall = sum(1 for r in comparison_rows if r['delta_recall10'] < 0)
ties_recall = len(comparison_rows) - wins_recall - losses_recall

print('=== Overall (Per-query mean) ===')
for k, v in summary.items():
    print(f'{k}: {v:.4f}')

print('\n=== Query-level comparison counts ===')
print(f'nDCG@10:   win={wins_ndcg}, loss={losses_ndcg}, tie={ties_ndcg}')
print(f'Recall@10: win={wins_recall}, loss={losses_recall}, tie={ties_recall}')


=== Overall (Per-query mean) ===
bm25_mean_ndcg10: 0.4364
hyde_mean_ndcg10: 0.5512
bm25_mean_recall10: 0.1285
hyde_mean_recall10: 0.1433

=== Query-level comparison counts ===
nDCG@10:   win=28, loss=15, tie=0
Recall@10: win=22, loss=15, tie=6


In [7]:
def topk_relevant_with_rank(ranked_docids, qrels_for_qid, k=10):
    # Returns [(docid, rank, rel), ...]
    out = []
    for rank, docid in enumerate(ranked_docids[:k], start=1):
        rel = qrels_for_qid.get(docid, 0)
        if rel > 0:
            out.append((docid, rank, rel))
    return out


def build_ranking_difference(qid, k=10):
    qrels_for_qid = qrels[qid]
    b_topk = runs[BASELINE_NAME][qid][:k]
    h_topk = runs[HYDE_NAME][qid][:k]

    b_rel = topk_relevant_with_rank(runs[BASELINE_NAME][qid], qrels_for_qid, k=k)
    h_rel = topk_relevant_with_rank(runs[HYDE_NAME][qid], qrels_for_qid, k=k)

    b_rel_docs = {docid for docid, _, _ in b_rel}
    h_rel_docs = {docid for docid, _, _ in h_rel}

    return {
        'qid': qid,
        'query': topics.get(qid, ''),
        'baseline_topk': b_topk,
        'hyde_topk': h_topk,
        'baseline_relevant_topk': b_rel,
        'hyde_relevant_topk': h_rel,
        'lost_relevant_docs_in_hyde_topk': sorted(b_rel_docs - h_rel_docs),
        'gained_relevant_docs_in_hyde_topk': sorted(h_rel_docs - b_rel_docs),
    }


def print_failure_detail(qid, k=10):
    diff = build_ranking_difference(qid, k=k)
    print(f"qid={qid} | query={diff['query']}")
    print(f"  baseline relevant@{k}: {diff['baseline_relevant_topk']}")
    print(f"  hyde     relevant@{k}: {diff['hyde_relevant_topk']}")
    print(f"  lost relevant docs in hyde top{k}: {diff['lost_relevant_docs_in_hyde_topk']}")
    print(f"  gained relevant docs in hyde top{k}: {diff['gained_relevant_docs_in_hyde_topk']}")


## Step 3: Identify Failure Cases

Failure case definition: **HyDE is worse than baseline on either nDCG@10 or Recall@10 for that query**.


In [8]:
failure_rows = [
    row for row in comparison_rows
    if (row['delta_ndcg10'] < 0) or (row['delta_recall10'] < 0)
]

# Sort by worst nDCG@10 drop first, then by Recall@10 drop.
failure_rows = sorted(failure_rows, key=lambda r: (r['delta_ndcg10'], r['delta_recall10']))

print(f'Total failure queries: {len(failure_rows)} / {len(comparison_rows)}')

if pd is not None:
    display(pd.DataFrame(failure_rows).head(20))
else:
    for row in failure_rows[:20]:
        print(
            row['qid'],
            f"delta_ndcg10={row['delta_ndcg10']:.4f}",
            f"delta_recall10={row['delta_recall10']:.4f}",
            row['query'],
        )

TOP_N_TO_INSPECT = 10
print(f'\nDetailed ranking differences for top {TOP_N_TO_INSPECT} failure queries:')
for row in failure_rows[:TOP_N_TO_INSPECT]:
    print('\n' + '=' * 100)
    print(
        f"qid={row['qid']} | "
        f"delta_ndcg10={row['delta_ndcg10']:.4f}, "
        f"delta_recall10={row['delta_recall10']:.4f}"
    )
    print_failure_detail(row['qid'], k=TOP_K)


Total failure queries: 20 / 43


,qid,query,baseline_ndcg10,hyde_ndcg10,delta_ndcg10,baseline_recall10,hyde_recall10,delta_recall10
0,1037798,who is robert gray,0.381611,0.000000,-0.381611,0.076923,0.000000,-0.076923
1,148538,difference between rn and bsn,0.368889,0.030589,-0.338300,0.049505,0.019802,-0.029703
2,19335,anthropological definition of environment,0.605109,0.272580,-0.332529,0.200000,0.100000,-0.100000
3,489204,right pelvic pain causes,0.314115,0.021942,-0.292173,0.041667,0.010417,-0.031250
4,855410,what is theraderm used for,0.960413,0.692615,-0.267798,1.000000,1.000000,0.000000
5,47923,axon terminals or synaptic knob definition,0.356604,0.098995,-0.257609,0.089286,0.026786,-0.062500
6,87452,causes of military suicide,0.320127,0.078418,-0.241710,0.098765,0.024691,-0.074074
7,87181,causes of left ventricular hypertrophy,0.589665,0.349915,-0.239750,0.096386,0.060241,-0.036145
8,104861,cost of interior concrete flooring,0.810750,0.593060,-0.217690,0.056738,0.042553,-0.014184
9,156493,do goldfish grow,0.848748,0.651123,-0.197625,0.075188,0.052632,-0.022556



Detailed ranking differences for top 10 failure queries:

qid=1037798 | delta_ndcg10=-0.3816, delta_recall10=-0.0769
qid=1037798 | query=who is robert gray
  baseline relevant@10: [('3641634', 1, 3)]
  hyde     relevant@10: []
  lost relevant docs in hyde top10: ['3641634']
  gained relevant docs in hyde top10: []

qid=148538 | delta_ndcg10=-0.3383, delta_recall10=-0.0297
qid=148538 | query=difference between rn and bsn
  baseline relevant@10: [('1950974', 1, 2), ('1950976', 2, 1), ('4185812', 3, 1), ('7407803', 4, 3), ('231455', 10, 1)]
  hyde     relevant@10: [('332401', 8, 1), ('615407', 9, 1)]
  lost relevant docs in hyde top10: ['1950974', '1950976', '231455', '4185812', '7407803']
  gained relevant docs in hyde top10: ['332401', '615407']

qid=19335 | delta_ndcg10=-0.3325, delta_recall10=-0.1000
qid=19335 | query=anthropological definition of environment
  baseline relevant@10: [('8412684', 1, 3), ('3175481', 2, 3), ('1729', 6, 2), ('8412681', 10, 2)]
  hyde     relevant@10: [('

In [ ]:
# Save outputs for reporting and further analysis.
OUT_DIR = DL19_DIR / 'error_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_out = OUT_DIR / 'comparison_per_query.jsonl'
failure_out = OUT_DIR / 'failure_cases.jsonl'

with comparison_out.open('w', encoding='utf-8') as f:
    for row in comparison_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

with failure_out.open('w', encoding='utf-8') as f:
    for row in failure_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved:')
print(f'  - {comparison_out}')
print(f'  - {failure_out}')
